# WinBleTouch

WinBleTouch lets a Windows PC send touchscreen input to an iPhone over Bluetooth.

> **About this notebook**
>
> The main project is the touch-control API. This notebook demonstrates a few ways to use it, rather than defining what it can do.
>
> GIFs and screenshots use ToolNaru to show the reported touches.
>
> Other applications may interpret the same input differently.

Run one cell at a time. **Do not use Run All.**
Each cell can send real touch input to your phone.

Read the [README.md](README.md) for the prerequisites.


In [119]:
import importlib
import time
import winbletouch

importlib.reload(winbletouch)

WinTouch = winbletouch.WinTouch

print([name for name in dir(WinTouch) if not name.startswith("_")])

['close', 'contact', 'ping', 'release', 'send', 'status']


## Tap

Start with one touch at the centre of the screen.
`contact()` sets its position, `send()` transmits the touch report,
and `release()` lifts the finger.


In [170]:
with WinTouch() as t:
    t.contact(5000, 5000)
    t.send()
    time.sleep(.5)
    print(t.release())

ok release-all


<img src="assets/01_tap.png" width="300">

## Swipe

A swipe builds on the same operations.
Repeated `contact()` calls move the active touch, and each `send()`
reports its new position. `release()` ends the swipe.


In [171]:
with WinTouch() as t:
    steps = 24
    for i in range(steps + 1):
        y = 6000 + (4000-6000)*i/steps
        t.contact(5000, y)
        t.send()
        time.sleep(1/steps)
    print(t.release())

ok release-all


<img src="assets/02_single_swipe.gif" width="300">

## Contact IDs

The tap and swipe examples use the default contact `id=0`. To track multiple fingers at once, each active contact needs its own ID.

The image below shows ToolNaru registering ten contacts simultaneously. The API supports ten contacts by default, but this capacity can be configured when starting the host.

<img src="assets\phase-01-all-move.png" width="150">

**Note the API can report multiple active contacts succsessfully, but iOS and the receiving application determine what those contacts actually do.**

For example, well-known multi-finger gestures like pinching to zoom or rotating an image rely on tracking separate contact IDs.

### Pinch gesture

Here, two active contacts move closer together or farther apart.
Both positions are updated before each `send()`.


In [174]:
zoom = 0  # set 0 for zoom out

with WinTouch() as t:
    steps = 50
    for i in range(steps + 1):
        if zoom == 1: spread = 500 + (1500 - 500) * i / steps
        else: spread = 1500 - (1500 - 500) * i / steps
        t.contact(5000, 5000 - spread, id=0)
        t.contact(5000, 5000 + spread, id=1)
        t.send()
        time.sleep(0.02)
    print(t.release())

ok release-all


<img src="assets/05_two_finger_pinch.gif" width="300">

### Rotation gesture

This example moves the two contacts along opposite arcs.
The vertical movement includes an aspect-ratio adjustment because
screen coordinates do not necessarily have equal physical scale.


In [175]:
import math

RADIUS = 1200
STEPS = 40
DELAY = 0.02

ASPECT_RATIO = 0.4612676

with WinTouch() as t:
    cx, cy = 5000, 5000

    for i in range(STEPS + 1):
        angle = math.pi / 2 * i / STEPS

        dx = RADIUS * math.cos(angle)
        dy = RADIUS * ASPECT_RATIO * math.sin(angle)

        t.contact(cx + round(dx), cy + round(dy), id=0)
        t.contact(cx - round(dx), cy - round(dy), id=1)

        t.send()
        time.sleep(DELAY)

    print(t.release())

ok release-all


<img src="assets/06_two_finger_rotation.gif" width="300">

## System gestures

System navigation uses the same contact operations as the earlier examples.
The difference is the gesture path: iOS can use the starting edge,
movement and timing to recognise Home, App Switcher and other actions.


In [161]:
# Home: quick flick up from the bottom edge
with WinTouch() as t:
    t.contact(5000, 10000)
    t.send()
    time.sleep(0.04)
    for y in range(9980, 9799, -20):
        t.contact(5000, y)
        t.send()
        time.sleep(0.008)
    for i in range(1, 18):
        t.contact(5000, 9800 + (5500 - 9800) * i / 17)
        t.send()
        time.sleep(0.008)
    print(t.release())

ok release-all


In [164]:
# App Switcher: slow swipe up from bottom edge, then hold
with WinTouch() as t:
    t.contact(5000, 10000)
    t.send()
    time.sleep(0.08)
    for y in range(9980, 9799, -20):
        t.contact(5000, y)
        t.send()
        time.sleep(0.016)
    for i in range(1, 29):
        t.contact(5000, 9800 + (7000 - 9800) * i / 28)
        t.send()
        time.sleep(0.016)
    time.sleep(0.80)
    print(t.release())

ok release-all


In [168]:
# Notification Center: swipe down from top-left
with WinTouch() as t:
    steps = 25
    for i in range(steps + 1):
        y = 0 + (8000 - 0) * i / steps
        t.contact(2000, y)
        t.send()
        time.sleep(0.1/ steps)
    print(t.release())

ok release-all


In [169]:
# Control Center: swipe down from top-right
with WinTouch() as t:
    steps = 25
    for i in range(steps + 1):
        y = 0 + (8000 - 0) * i / steps
        t.contact(8000, y)
        t.send()
        time.sleep(0.1/steps)
    print(t.release())

ok release-all


## Troubleshooting

Check the connection status and release any contacts left active after testing.


In [ ]:
with WinTouch() as touch:
    print(touch.status())
    print(touch.release())